In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# Source: raw GH Archive JSON files in the Volume
VOLUME_PATH = "/Volumes/gharchive_dev/raw/files/*.json.gz"
BRONZE_TABLE = "gharchive_dev.v1_pyspark.gharchive_bronze"

# Read raw JSON files and add ingestion metadata
df_raw = (
    spark.read
    .format("json")
    .option("multiLine", "false")
    .load(VOLUME_PATH)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

# Filter out files already ingested (deduplication)
if spark.catalog.tableExists(BRONZE_TABLE):
    existing_files = (
        spark.read.table(BRONZE_TABLE)
        .select("_source_file")
        .distinct()
    )
    df_bronze = df_raw.join(existing_files, on="_source_file", how="left_anti")
    new_file_count = df_bronze.select("_source_file").distinct().count()
    print(f"New files to ingest: {new_file_count}")
else:
    df_bronze = df_raw
    print("Table does not exist yet. Ingesting all files.")

# Append only new data to Delta table
if df_bronze.head(1):
    df_bronze.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
    print(f"Bronze table appended: {BRONZE_TABLE}")
else:
    print("No new files to ingest. Skipping write.")

print(f"Total row count: {spark.read.table(BRONZE_TABLE).count()}")